In [89]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import LabelEncoder


In [90]:
df = pd.read_csv('GDSC2-dataset.csv')
df2 = pd.read_csv('GDSC_DATASET.csv')
df3 = pd.read_csv('Compounds-annotation.csv')
df4 = pd.read_excel('Cell_Lines_Details.xlsx', engine='openpyxl')

c:\Users\allan\anaconda3\Lib\site-packages\openpyxl\worksheet\_read_only.py:85: UserWarning: Unknown extension is not supported and will be removed
  for idx, row in parser.parse():


In [91]:
df.head()
df2.head()
df3.head()
df4.head()

,Sample Name,COSMIC identifier,Whole Exome Sequencing (WES),Copy Number Alterations (CNA),Gene Expression,Methylation,Drug\nResponse,GDSC\nTissue descriptor 1,GDSC\nTissue\ndescriptor 2,Cancer Type\n(matching TCGA label),Microsatellite \ninstability Status (MSI),Screen Medium,Growth Properties
0,A253,906794.0,Y,Y,Y,Y,Y,aero_dig_tract,head and neck,NaN,MSS/MSI-L,D/F12,Adherent
1,BB30-HNC,753531.0,Y,Y,Y,Y,Y,aero_dig_tract,head and neck,HNSC,MSS/MSI-L,D/F12,Adherent
2,BB49-HNC,753532.0,Y,Y,Y,Y,Y,aero_dig_tract,head and neck,HNSC,MSS/MSI-L,D/F12,Adherent
3,BHY,753535.0,Y,Y,Y,Y,Y,aero_dig_tract,head and neck,HNSC,MSS/MSI-L,D/F12,Adherent
4,BICR10,1290724.0,Y,Y,Y,Y,Y,aero_dig_tract,head and neck,HNSC,MSS/MSI-L,D/F12,Adherent


In [92]:
# Confirm TARGET and TARGET_PATHWAY are already in df2
for col in ["TARGET", "TARGET_PATHWAY"]:
    n_valid = df2[col].notna().sum()
    print(f"[Step 2] {col}: {n_valid:,} populated values ({n_valid/len(df2)*100:.1f}% coverage) ✓")

[Step 2] TARGET: 214,880 populated values (88.8% coverage) ✓
[Step 2] TARGET_PATHWAY: 242,035 populated values (100.0% coverage) ✓


In [93]:
# STEP 2 — Drops 

# drop rows with null value on TCGA_DESC
rows_before = len(df2)
df2 = df2.dropna(subset=["TCGA_DESC"])
rows_dropped = rows_before - len(df2)
print(f"Dropped {rows_dropped} rows with null TCGA_DESC "
      f"({rows_dropped/rows_before*100:.2f}% of data). "
      f"Remaining rows: {len(df2):,}")

Dropped 1067 rows with null TCGA_DESC (0.44% of data). Remaining rows: 240,968


In [94]:
# Drops the Unclassified category from TCGA
rows_before_unclass = len(df2)
df2 = df2[df2["TCGA_DESC"] != "UNCLASSIFIED"]
rows_dropped_unclass = rows_before_unclass - len(df2)
print(f"Dropped {rows_dropped_unclass:,} UNCLASSIFIED rows "
      f"({rows_dropped_unclass/rows_before_unclass*100:.2f}% of data). "
      f"Remaining rows: {len(df2):,}")

Dropped 45,690 UNCLASSIFIED rows (18.96% of data). Remaining rows: 195,278


In [95]:
# drop rows where all 5 cell line columns are simultaneously UNKNOWN
DF2_CELL_LINE_COLS = [
    "Microsatellite instability Status (MSI)",
    "GDSC Tissue descriptor 1",
    "GDSC Tissue descriptor 2",
    "Growth Properties",
    "Screen Medium",
]
all_unknown_mask = (df2[DF2_CELL_LINE_COLS] == "UNKNOWN").all(axis=1)
rows_before_failed = len(df2)
df2 = df2[~all_unknown_mask]
rows_dropped_failed = rows_before_failed - len(df2)
print(f"Dropped {rows_dropped_failed:,} rows where all 5 cell line columns "
      f"were simultaneously UNKNOWN (failed joins). "
      f"Remaining rows: {len(df2):,}")

Dropped 0 rows where all 5 cell line columns were simultaneously UNKNOWN (failed joins). Remaining rows: 195,278


In [96]:
# STEP 3 — Checks if LN_IC50 has no nulls and confirms the range/mean/std.
if df2["LN_IC50"].isna().sum() == 0:
    pass # clean – nothing to do
else:
    raise AssertionError("Unexpected nulls in LN_IC50!")

print(f"\n[EDA 1] LN_IC50 confirmed clean:")
print(f"  Range : {df2['LN_IC50'].min():.2f}  →  {df2['LN_IC50'].max():.2f}")
print(f"  Mean  : {df2['LN_IC50'].mean():.4f}   Std: {df2['LN_IC50'].std():.4f}")


[EDA 1] LN_IC50 confirmed clean:
  Range : -8.64  →  13.82
  Mean  : 2.8853   Std: 2.7534


In [97]:
# STEP 4 — Defines all 8 feature columns using the corrected exact names from df2 and the new columns from df3.
FEATURE_COLS = [
    "TCGA_DESC",                              # EDA 4: 33 cats, all kept
    "Microsatellite instability Status (MSI)", # EDA 5: exact name from df2
    "GDSC Tissue descriptor 1",               # EDA 5: exact name from df2
    "GDSC Tissue descriptor 2",               # EDA 5: exact name from df2
    "Growth Properties",                      # EDA 5: strong signal (eta²=0.027)
    "Screen Medium",                          # EDA 5: real signal (eta²=0.016)
    "TARGET",                                 # from Compounds-annotation.csv
    "TARGET_PATHWAY",                         # from Compounds-annotation.csv
]

TARGET_COL = "LN_IC50"

In [98]:
# STEP 5 — Validates every column exists
missing_cols = [c for c in FEATURE_COLS if c not in df2.columns]
if missing_cols:
    print(f"\n These columns were NOT found in df2: {missing_cols}")
    print("   Check column names with: df2.columns.tolist()")
    FEATURE_COLS = [c for c in FEATURE_COLS if c in df2.columns]
else:
    print(f"\n All {len(FEATURE_COLS)} feature columns confirmed present in df2")



 All 8 feature columns confirmed present in df2


In [99]:
# STEP 6 — Encodes all 8 features with LabelEncoder

df_model = df2[FEATURE_COLS + [TARGET_COL]].copy()

label_encoders = {}
encoding_summary = []

for col in FEATURE_COLS:
    # Fill any residual nulls with "UNKNOWN" before encoding, to avoid errors and preserve all rows.
    null_count = df_model[col].isna().sum()
    if null_count > 0:
        df_model[col] = df_model[col].fillna("UNKNOWN")
        print(f"  [{col}] Filled {null_count} nulls with 'UNKNOWN' before encoding")

    le = LabelEncoder()
    df_model[col + "_enc"] = le.fit_transform(df_model[col].astype(str))
    label_encoders[col] = le

    n_cats = df_model[col].nunique()
    encoding_summary.append({"Feature": col, "Unique Categories": n_cats,
                              "Encoded Column": col + "_enc"})

print("\n[EDA 4 & 5] Label Encoding Summary:")
enc_df = pd.DataFrame(encoding_summary)
print(enc_df.to_string(index=False))


  [Microsatellite instability Status (MSI)] Filled 10384 nulls with 'UNKNOWN' before encoding
  [GDSC Tissue descriptor 1] Filled 7934 nulls with 'UNKNOWN' before encoding
  [GDSC Tissue descriptor 2] Filled 7934 nulls with 'UNKNOWN' before encoding
  [Growth Properties] Filled 7934 nulls with 'UNKNOWN' before encoding
  [Screen Medium] Filled 7934 nulls with 'UNKNOWN' before encoding
  [TARGET] Filled 21911 nulls with 'UNKNOWN' before encoding

[EDA 4 & 5] Label Encoding Summary:
                                Feature  Unique Categories                              Encoded Column
                              TCGA_DESC                 31                               TCGA_DESC_enc
Microsatellite instability Status (MSI)                  3 Microsatellite instability Status (MSI)_enc
               GDSC Tissue descriptor 1                 18                GDSC Tissue descriptor 1_enc
               GDSC Tissue descriptor 2                 34                GDSC Tissue descriptor 2_enc

In [100]:
# STEP 7 — Make the X and Y for the splits
encoded_feature_cols = [c + "_enc" for c in FEATURE_COLS]

X = df_model[encoded_feature_cols].copy()
X.columns = FEATURE_COLS   # rename back to clean names for readability

y = df_model[TARGET_COL]

print(f"\n[Final Dataset] X shape: {X.shape}  |  y shape: {y.shape}")


[Final Dataset] X shape: (195278, 8)  |  y shape: (195278,)


In [101]:
# STEP 8 — View of the processed data
print("PROCESSED DATA PREVIEW (first 5 rows)")
preview_readable = df_model[FEATURE_COLS].copy()
preview_readable["LN_IC50 (target)"] = y.values
print(preview_readable.head().to_string())

print("ENCODED VALUE MAPPINGS (first 5 values per feature)")
for col, le in label_encoders.items():
    mapping = dict(zip(le.classes_[:5], le.transform(le.classes_[:5])))
    print(f"  {col}: {mapping}{'...' if len(le.classes_) > 5 else ''}")


PROCESSED DATA PREVIEW (first 5 rows)
  TCGA_DESC Microsatellite instability Status (MSI) GDSC Tissue descriptor 1 GDSC Tissue descriptor 2 Growth Properties Screen Medium TARGET   TARGET_PATHWAY  LN_IC50 (target)
0        MB                               MSS/MSI-L           nervous_system          medulloblastoma          Adherent             R   TOP1  DNA replication         -1.463887
5      SKCM                               MSS/MSI-L                     skin                 melanoma          Adherent             R   TOP1  DNA replication         -1.235034
6      BLCA                                 UNKNOWN                  UNKNOWN                  UNKNOWN           UNKNOWN       UNKNOWN   TOP1  DNA replication         -2.632632
7      BLCA                               MSS/MSI-L        urogenital_system                  Bladder          Adherent         D/F12   TOP1  DNA replication         -2.963191
8      BLCA                               MSS/MSI-L        urogenital_system      